# Lesson 29. Python with 关键字

&emsp;&emsp;在 Python 编程中，资源管理是一个重要但容易被忽视的环节。**with 关键字** 为我们提供了一种优雅的方式来处理文件操作、数据库连接等需要明确释放资源的场景。<br>
&emsp;&emsp;with 是 Python 中的一个关键字，用于上下文管理协议（Context Management Protocol）。它简化了资源管理代码，特别是那些需要明确释放或清理的资源（如文件、网络连接、数据库连接等）。

## 1. 为什么需要 with 语句？

&emsp;&emsp;传统资源管理的问题，我们先看一个典型的文件操作示例：

In [1]:
file = open('/etc/hosts', 'r')
try:
    context = file.read()
finally:
    file.close()

print(context)

127.0.0.1	localhost
127.0.1.1	datasci-ml.lab.example.com	datasci-ml

# The following lines are desirable for IPv6 capable hosts
::1     ip6-localhost ip6-loopback
fe00::0 ip6-localnet
ff00::0 ip6-mcastprefix
ff02::1 ip6-allnodes
ff02::2 ip6-allrouters

192.168.110.115  kernel-aiops.lab.example.com  kernel-aiops
192.168.0.107  kernel-aiops.lab.example.com  kernel-aiops



&emsp;&emsp;这种写法存在几个问题：

- 容易忘记关闭资源：如果没有 try-finally 块，可能会忘记调用 close()；
- 代码冗长：简单的文件操作需要多行代码；
- 异常处理复杂：需要手动处理可能出现的异常。

&emsp;&emsp;with 语句通过 **上下文管理协议（Context Management Protocol）** 解决了这些问题：

- 自动资源释放：确保资源在使用后被正确关闭；
- 代码简洁：减少样板代码；
- 异常安全：即使在代码块中发生异常，资源也会被正确释放；
- 可读性强：明确标识资源的作用域。

## 2. with 语句的基本语法

### 2.1 基本用法

&emsp;&emsp;语法格式：

```python
with expression [as variable]:
    # 代码块
```

- expression 返回一个支持上下文管理协议的对象；
- as variable 是可选的，用于将表达式结果赋值给变量；
- 代码块执行完毕后，自动调用清理方法。

### 2.2 文件操作示例

In [2]:
with open('/etc/hosts', 'r') as file:
    context = file.read()
    print(context)

127.0.0.1	localhost
127.0.1.1	datasci-ml.lab.example.com	datasci-ml

# The following lines are desirable for IPv6 capable hosts
::1     ip6-localhost ip6-loopback
fe00::0 ip6-localnet
ff00::0 ip6-mcastprefix
ff02::1 ip6-allnodes
ff02::2 ip6-allrouters

192.168.110.115  kernel-aiops.lab.example.com  kernel-aiops
192.168.0.107  kernel-aiops.lab.example.com  kernel-aiops



&emsp;&emsp;这段代码等价于前面的 `try-finally` 实现，但更加简洁明了。

## 3. with 语句的工作原理

**上下文管理协议：**

&emsp;&emsp;with 语句背后是 Python 的上下文管理协议，该协议要求对象实现两个方法：

- `__enter__()`：进入上下文时调用，返回值赋给 as 后的变量；
- `__exit__()`：退出上下文时调用，处理清理工作。

**执行流程：**

<center><img src="images/python-with-keyword.png" width=60%></center>

**异常处理机制：**

&emsp;&emsp;`__exit__()` 方法接收三个参数：

- exc_type：异常类型
- exc_val：异常值
- exc_tb：异常追踪信息

&emsp;&emsp;如果 `__exit__()` 返回 True，则表示异常已被处理，不会继续传播；返回 False 或 None，异常会继续向外传播。

## 4. 实际应用场景

### 4.1 文件操作

In [6]:
with open('/etc/os-release', 'r') as infile, open('/tmp/os-release', 'w') as outfile:
    context = infile.read()
    outfile.write(context.upper())

with open('/tmp/os-release', 'r') as backup:
    print(backup.read())

PRETTY_NAME="UBUNTU 24.04.4 LTS"
NAME="UBUNTU"
VERSION_ID="24.04"
VERSION="24.04.4 LTS (NOBLE NUMBAT)"
VERSION_CODENAME=NOBLE
ID=UBUNTU
ID_LIKE=DEBIAN
HOME_URL="HTTPS://WWW.UBUNTU.COM/"
SUPPORT_URL="HTTPS://HELP.UBUNTU.COM/"
BUG_REPORT_URL="HTTPS://BUGS.LAUNCHPAD.NET/UBUNTU/"
PRIVACY_POLICY_URL="HTTPS://WWW.UBUNTU.COM/LEGAL/TERMS-AND-POLICIES/PRIVACY-POLICY"
UBUNTU_CODENAME=NOBLE
LOGO=UBUNTU-LOGO



## 4.2 多线程互斥锁（Mutex Lock）❓

&emsp;&emsp;Python 中默认使用互斥锁（Mutex Lock）实现多进程间对资源的访问控制：

In [9]:
import threading
import time
import random

class BankAccount:
    """无锁版本 - 存在竞态条件"""
    def __init__(self, balance=1000):
        self.balance = balance
    
    def withdraw_unsafe(self, amount):
        """不安全取款：检查余额和扣款之间可能被中断"""
        if self.balance >= amount:    # 步骤1: 检查
            time.sleep(0.001)         # 模拟耗时（让出CPU）
            self.balance -= amount    # 步骤2: 扣款
            return True
        return False

class SafeBankAccount:
    """有锁版本 - 线程安全"""
    def __init__(self, balance=1000):
        self.balance = balance
        self.lock = threading.Lock()    # 保护状态的一致性
    
    def withdraw_safe(self, amount):
        with self.lock:                 # 获取锁，独占访问
            if self.balance >= amount:  # 检查+扣款成为原子操作
                time.sleep(0.001)
                self.balance -= amount
                return True
            return False

# ============ 测试：模拟并发取款 ============

def test_unsafe():
    print("=== 无锁版本（预期会出现超支）===")
    account = BankAccount(1000)
    
    def try_withdraw():
        for _ in range(100):
            account.withdraw_unsafe(10)  # 每次取10元
    
    threads = [threading.Thread(target=try_withdraw) for _ in range(20)]
    for t in threads: t.start()
    for t in threads: t.join()
    
    print(f"最终余额: {account.balance} 元")
    print(f"理论最小值: {1000 - 20*100*10} = -19000 元（允许超支）")
    print(f"实际是否超支: {'是' if account.balance < 0 else '否'}\n")

def test_safe():
    print("=== 有锁版本（不会超支）===")
    account = SafeBankAccount(1000)
    
    def try_withdraw():
        for _ in range(100):
            account.withdraw_safe(10)
    
    threads = [threading.Thread(target=try_withdraw) for _ in range(20)]
    for t in threads: t.start()
    for t in threads: t.join()
    
    print(f"最终余额: {account.balance} 元")
    print(f"是否超支: {'是' if account.balance < 0 else '否'}")
    print(f"成功阻止了 {20*100 - (1000-account.balance)//10} 次超额取款尝试")

if __name__ == "__main__":
    test_unsafe()
    test_safe()

=== 无锁版本（预期会出现超支）===
最终余额: -160 元
理论最小值: -19000 = -19000 元（允许超支）
实际是否超支: 是

=== 有锁版本（不会超支）===
最终余额: 0 元
是否超支: 否
成功阻止了 1900 次超额取款尝试


## 4.3 临时修改系统状态